<a href="https://colab.research.google.com/github/wonsoleun/09/blob/main/Untitled2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

graph = {
    '트리토리아진': [('담금', 299), ('빠오즈푸', 208), ('다온카츠', 249)],
    '담금':        [('트리토리아진', 299), ('빠오즈푸', 141), ('땀땀', 173)],
    '빠오즈푸':    [('트리토리아진', 208), ('익선취향', 119), ('담금', 141)],
    '다온카츠':    [('트리토리아진', 249), ('익선취향', 164)],
    '익선취향':    [('다온카츠', 164), ('마복림할머니집', 52), ('빠오즈푸', 119)],
    '마복림할머니집': [('익선취향', 52), ('이쿠', 222)],
    '이쿠':        [('마복림할머니집', 222), ('땀땀', 120)],
    '땀땀':        [('이쿠', 120), ('담금', 173)],
}

edges = [
    ('트리토리아진', '담금',           299),
    ('트리토리아진', '빠오즈푸',       208),
    ('트리토리아진', '다온카츠',       249),
    ('다온카츠',     '익선취향',       164),
    ('익선취향',     '마복림할머니집',  52),
    ('익선취향',     '빠오즈푸',       119),
    ('마복림할머니집','이쿠',          222),
    ('빠오즈푸',     '담금',           141),
    ('이쿠',         '땀땀',           120),
    ('땀땀',         '담금',           173),
]

nodes = list(graph.keys())

import matplotlib
matplotlib.use('TkAgg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.animation as animation
import matplotlib.font_manager as fm
import os


def find_korean_font():
    candidates = [
        'C:/Windows/Fonts/malgun.ttf',
        'C:/Windows/Fonts/gulim.ttc',
        '/System/Library/Fonts/AppleSDGothicNeo.ttc',
        '/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
        '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc',
    ]
    for path in candidates:
        if os.path.exists(path):
            return path
    return None

font_path = find_korean_font()
_fp = fm.FontProperties(fname=font_path) if font_path else fm.FontProperties()
plt.rcParams['axes.unicode_minus'] = False


pos = {
    '트리토리아진':    (-2.5,  1.5),
    '다온카츠':        (-1.0,  0.0),
    '익선취향':        ( 0.5,  0.0),
    '마복림할머니집':  ( 0.5, -1.8),
    '이쿠':            ( 2.0, -1.8),
    '땀땀':            ( 3.2, -0.5),
    '담금':            ( 3.2,  1.5),
    '빠오즈푸':        ( 1.2,  1.5),
}


edges = [
    ('마복림할머니집', '빠오즈푸',      70),
    ('트리토리아진', '빠오즈푸',       208),
    ('트리토리아진', '다온카츠',       249),
    ('다온카츠',     '익선취향',       164),
    ('익선취향',     '마복림할머니집',  52),
    ('익선취향',     '빠오즈푸',       119),
    ('마복림할머니집','이쿠',          222),
    ('빠오즈푸',     '담금',           141),
    ('이쿠',         '땀땀',           120),
    ('땀땀',         '담금',           173),
    ('빠오즈푸',        '땀땀',         115)
]


graph = {node: {} for node in pos}
for u, v, w in edges:
    graph[u][v] = w
    graph[v][u] = w

nodes = list(pos.keys())


def dfs_steps(graph, start):
    visited, visited_set = [], set()
    stack = [(start, None)]
    tree_edges = []
    steps = []

    while stack:
        node, parent = stack.pop()
        if node in visited_set:
            continue
        visited.append(node)
        visited_set.add(node)
        if parent is not None:
            tree_edges.append((parent, node))

        neighbors = sorted(
            [(v, w) for v, w in graph[node].items() if v not in visited_set],
            key=lambda x: x[1], reverse=True
        )
        for v, _ in neighbors:
            stack.append((v, node))

        steps.append({
            'visited':    list(visited),
            'tree_edges': list(tree_edges),
            'current':    node,
            'stack':      [s[0] for s in stack if s[0] not in visited_set],
        })

    return steps

steps = dfs_steps(graph, '트리토리아진')
all_steps = steps + [steps[-1]] * 30


C_BG        = '#1a1a2e'
C_EDGE_DEF  = '#4a4a6a'
C_EDGE_TREE = '#e94560'
C_NODE_DEF  = '#16213e'
C_NODE_VIS  = '#0f3460'
C_NODE_CUR  = '#e94560'
C_NODE_STK  = '#533483'
C_TEXT      = '#eaeaea'
C_WT        = '#a0a0c0'


fig = plt.figure(figsize=(14, 8), facecolor=C_BG)
ax_graph = fig.add_axes([0.0, 0.12, 0.72, 0.82])
ax_info  = fig.add_axes([0.73, 0.12, 0.25, 0.82])

NODE_RADIUS = 0.18

def draw_frame(i):
    ax_graph.cla()
    ax_info.cla()
    for ax in [ax_graph, ax_info]:
        ax.set_facecolor(C_BG)
        ax.axis('off')

    step        = all_steps[i]
    visited     = step['visited']
    tree_edges  = step['tree_edges']
    current     = step['current']
    stack_nodes = step['stack']


    xs = [p[0] for p in pos.values()]
    ys = [p[1] for p in pos.values()]
    ax_graph.set_xlim(min(xs) - 0.8, max(xs) + 0.8)
    ax_graph.set_ylim(min(ys) - 0.8, max(ys) + 0.8)
    ax_graph.set_aspect('equal')


    for u, v, w in edges:
        x1, y1 = pos[u]
        x2, y2 = pos[v]
        is_tree = (u, v) in tree_edges or (v, u) in tree_edges
        color = C_EDGE_TREE if is_tree else C_EDGE_DEF
        lw    = 3.0 if is_tree else 1.2
        ax_graph.plot([x1, x2], [y1, y2], color=color, linewidth=lw, zorder=1)


        mx, my = (x1 + x2) / 2, (y1 + y2) / 2
        ax_graph.text(mx, my, f'{w}분', color=C_WT, fontsize=7.5,
                      ha='center', va='center', fontproperties=_fp, zorder=3,
                      bbox=dict(boxstyle='round,pad=0.2', fc=C_BG, ec='none', alpha=0.8))


    for node in nodes:
        x, y = pos[node]
        if node == current:
            color = C_NODE_CUR
        elif node in visited:
            color = C_NODE_VIS
        elif node in stack_nodes:
            color = C_NODE_STK
        else:
            color = C_NODE_DEF

        circle = plt.Circle((x, y), NODE_RADIUS, color=color,
                             ec='white', linewidth=1.5, zorder=2)
        ax_graph.add_patch(circle)
        ax_graph.text(x, y, node, color=C_TEXT, fontsize=8,
                      ha='center', va='center', fontproperties=_fp, zorder=4,
                      fontweight='bold' if node == current else 'normal')

    ax_graph.set_title('DFS — 음식점 방문 경로', color=C_TEXT,
                       fontsize=14, fontweight='bold', pad=8, fontproperties=_fp)


    ax_info.set_xlim(0, 1)
    ax_info.set_ylim(0, 1)

    legend_items = [
        (C_NODE_CUR, '현재 방문 중'),
        (C_NODE_VIS, '방문 완료'),
        (C_NODE_STK, '스택 대기'),
        (C_NODE_DEF, '미방문'),
    ]
    ax_info.text(0.05, 0.97, '범례', color=C_TEXT, fontsize=10,
                 fontweight='bold', va='top', fontproperties=_fp)
    for k, (color, label) in enumerate(legend_items):
        y = 0.91 - k * 0.065
        ax_info.add_patch(plt.Circle((0.08, y), 0.028, color=color))
        ax_info.text(0.18, y, label, color=C_TEXT, fontsize=8.5,
                     va='center', fontproperties=_fp)

    ax_info.text(0.05, 0.63, f'방문 순서 ({len(visited)}/{len(nodes)})',
                 color=C_TEXT, fontsize=10, fontweight='bold', va='top', fontproperties=_fp)
    for k, node in enumerate(visited):
        y = 0.57 - k * 0.058
        if y < 0.02:
            break
        color = C_NODE_CUR if node == current else C_NODE_VIS
        ax_info.text(0.05, y, f'{k+1}. {node}', color=color,
                     fontsize=8, va='top', fontproperties=_fp)

    ax_info.text(0.05, 0.07, '스택 (top →)', color=C_TEXT, fontsize=8.5,
                 fontweight='bold', va='bottom', fontproperties=_fp)
    stack_str = ' → '.join(reversed(stack_nodes)) if stack_nodes else '(비어있음)'
    ax_info.text(0.05, 0.03, stack_str, color='#aaaaff', fontsize=7.5,
                 va='bottom', fontproperties=_fp)

    fig.text(0.01, 0.01, f'Step {min(i+1, len(steps))}/{len(steps)}',
             color='#666688', fontsize=8, ha='left')


ani = animation.FuncAnimation(
    fig, draw_frame,
    frames=len(all_steps),
    interval=900,
    repeat=False
)

plt.show()

In [ ]:
import os
import matplotlib
matplotlib.use('TkAgg')  # 환경에 맞게 변경 (필요시 복원)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.animation as animation
import matplotlib.font_manager as fm
from collections import deque

# ── 한글 폰트 설정 ──────────────────────────────────────────
def find_korean_font():
    candidates = [
        'C:/Windows/Fonts/malgun.ttf',
        'C:/Windows/Fonts/gulim.ttc',
        '/System/Library/Fonts/AppleSDGothicNeo.ttc',
        '/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
        '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc',
    ]
    for path in candidates:
        if os.path.exists(path):
            return path
    return None

font_path = find_korean_font()
_fp = fm.FontProperties(fname=font_path) if font_path else fm.FontProperties()
plt.rcParams['axes.unicode_minus'] = False

# ── 그래프 정의 ──────────────────────────────────────────────
pos = {
    '트리토리아진':    (-2.5,  1.5),
    '다온카츠':        (-1.0,  0.0),
    '익선취향':        ( 0.5,  0.0),
    '마복림할머니집':  ( 0.5, -1.8),
    '이쿠':            ( 2.0, -1.8),
    '땀땀':            ( 3.2, -0.5),
    '담금':            ( 3.2,  1.5),
    '빠오즈푸':        ( 1.2,  1.5),
}

edges = [
    ('마복림할머니집', '빠오즈푸',      70),
    ('트리토리아진', '빠오즈푸',       208),
    ('트리토리아진', '다온카츠',       249),
    ('다온카츠',     '익선취향',       164),
    ('익선취향',     '마복림할머니집',  52),
    ('익선취향',     '빠오즈푸',       119),
    ('마복림할머니집','이쿠',          222),
    ('빠오즈푸',     '담금',           141),
    ('이쿠',         '땀땀',           120),
    ('땀땀',         '담금',           173),
    ('빠오즈푸',        '땀땀',         115)
]

graph = {node: {} for node in pos}
for u, v, w in edges:
    graph[u][v] = w
    graph[v][u] = w

nodes = list(pos.keys())

# ── BFS 스텝 기록 ────────────────────────────────────────────
def bfs_steps(graph, start):
    visited, visited_set = [], set()
    queue = deque([(start, None)])
    tree_edges = []
    steps = []

    while queue:
        node, parent = queue.popleft()
        if node in visited_set:
            continue
        visited.append(node)
        visited_set.add(node)
        if parent is not None:
            tree_edges.append((parent, node))

        # 인접 노드를 가중치 오름차순으로 정렬 (가까운 가중치부터 큐에 삽입)
        neighbors = sorted(
            [(v, w) for v, w in graph[node].items() if v not in visited_set],
            key=lambda x: x[1]
        )
        for v, _ in neighbors:
            queue.append((v, node))

        # 시각화 화면 표시용 큐 노드 순서 정리 (중복 제거 및 선입선출 유지)
        queue_nodes = []
        seen = set()
        for q in queue:
            if q[0] not in visited_set and q[0] not in seen:
                queue_nodes.append(q[0])
                seen.add(q[0])

        steps.append({
            'visited':    list(visited),
            'tree_edges': list(tree_edges),
            'current':    node,
            'queue':      queue_nodes,
        })

    return steps

steps = bfs_steps(graph, '트리토리아진')
all_steps = steps + [steps[-1]] * 30

# ── 색상 ─────────────────────────────────────────────────────
C_BG        = '#1a1a2e'
C_EDGE_DEF  = '#4a4a6a'
C_EDGE_TREE = '#e94560'
C_NODE_DEF  = '#16213e'
C_NODE_VIS  = '#0f3460'
C_NODE_CUR  = '#e94560'
C_NODE_QUE  = '#533483'  # 큐 대기 노드 색상
C_TEXT      = '#eaeaea'
C_WT        = '#a0a0c0'

# ── Figure 설정 ──────────────────────────────────────────────
fig = plt.figure(figsize=(14, 8), facecolor=C_BG)
ax_graph = fig.add_axes([0.0, 0.12, 0.72, 0.82])
ax_info  = fig.add_axes([0.73, 0.12, 0.25, 0.82])

NODE_RADIUS = 0.18

def draw_frame(i):
    ax_graph.cla()
    ax_info.cla()
    for ax in [ax_graph, ax_info]:
        ax.set_facecolor(C_BG)
        ax.axis('off')

    step        = all_steps[i]
    visited     = step['visited']
    tree_edges  = step['tree_edges']
    current     = step['current']
    queue_nodes = step['queue']

    # x, y 범위 자동 설정
    xs = [p[0] for p in pos.values()]
    ys = [p[1] for p in pos.values()]
    ax_graph.set_xlim(min(xs) - 0.8, max(xs) + 0.8)
    ax_graph.set_ylim(min(ys) - 0.8, max(ys) + 0.8)
    ax_graph.set_aspect('equal')

    # 간선 그리기
    for u, v, w in edges:
        x1, y1 = pos[u]
        x2, y2 = pos[v]
        is_tree = (u, v) in tree_edges or (v, u) in tree_edges
        color = C_EDGE_TREE if is_tree else C_EDGE_DEF
        lw    = 3.0 if is_tree else 1.2
        ax_graph.plot([x1, x2], [y1, y2], color=color, linewidth=lw, zorder=1)

        # 가중치 라벨
        mx, my = (x1 + x2) / 2, (y1 + y2) / 2
        ax_graph.text(mx, my, f'{w}분', color=C_WT, fontsize=7.5,
                      ha='center', va='center', fontproperties=_fp, zorder=3,
                      bbox=dict(boxstyle='round,pad=0.2', fc=C_BG, ec='none', alpha=0.8))

    # 노드 그리기
    for node in nodes:
        x, y = pos[node]
        if node == current:
            color = C_NODE_CUR
        elif node in visited:
            color = C_NODE_VIS
        elif node in queue_nodes:
            color = C_NODE_QUE
        else:
            color = C_NODE_DEF

        circle = plt.Circle((x, y), NODE_RADIUS, color=color,
                             ec='white', linewidth=1.5, zorder=2)
        ax_graph.add_patch(circle)
        ax_graph.text(x, y, node, color=C_TEXT, fontsize=8,
                      ha='center', va='center', fontproperties=_fp, zorder=4,
                      fontweight='bold' if node == current else 'normal')

    ax_graph.set_title('BFS — 음식점 방문 경로', color=C_TEXT,
                       fontsize=14, fontweight='bold', pad=8, fontproperties=_fp)

    # ── 정보 패널 ──────────────────────────────────────────
    ax_info.set_xlim(0, 1)
    ax_info.set_ylim(0, 1)

    legend_items = [
        (C_NODE_CUR, '현재 방문 중'),
        (C_NODE_VIS, '방문 완료'),
        (C_NODE_QUE, '큐 대기 (Queue)'),
        (C_NODE_DEF, '미방문'),
    ]
    ax_info.text(0.05, 0.97, '범례', color=C_TEXT, fontsize=10,
                 fontweight='bold', va='top', fontproperties=_fp)
    for k, (color, label) in enumerate(legend_items):
        y = 0.91 - k * 0.065
        ax_info.add_patch(plt.Circle((0.08, y), 0.028, color=color))
        ax_info.text(0.18, y, label, color=C_TEXT, fontsize=8.5,
                     va='center', fontproperties=_fp)

    ax_info.text(0.05, 0.63, f'방문 순서 ({len(visited)}/{len(nodes)})',
                 color=C_TEXT, fontsize=10, fontweight='bold', va='top', fontproperties=_fp)
    for k, node in enumerate(visited):
        y = 0.57 - k * 0.058
        if y < 0.02:
            break
        color = C_NODE_CUR if node == current else C_NODE_VIS
        ax_info.text(0.05, y, f'{k+1}. {node}', color=color,
                     fontsize=8, va='top', fontproperties=_fp)

    ax_info.text(0.05, 0.07, '큐 상태 (Front → Rear)', color=C_TEXT, fontsize=8.5,
                 fontweight='bold', va='bottom', fontproperties=_fp)
    queue_str = ' → '.join(queue_nodes) if queue_nodes else '(비어있음)'
    ax_info.text(0.05, 0.03, queue_str, color='#aaaaff', fontsize=7.5,
                 va='bottom', fontproperties=_fp)

    fig.text(0.01, 0.01, f'Step {min(i+1, len(steps))}/{len(steps)}',
             color='#666688', fontsize=8, ha='left')


ani = animation.FuncAnimation(
    fig, draw_frame,
    frames=len(all_steps),
    interval=900,
    repeat=False
)

plt.show()

In [ ]:
import os
import matplotlib
matplotlib.use('TkAgg')  # 환경에 맞게 변경
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.animation as animation
import matplotlib.font_manager as fm

# ── 한글 폰트 설정 ──────────────────────────────────────────
def find_korean_font():
    candidates = [
        'C:/Windows/Fonts/malgun.ttf',
        'C:/Windows/Fonts/gulim.ttc',
        '/System/Library/Fonts/AppleSDGothicNeo.ttc',
        '/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
        '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc',
    ]
    for path in candidates:
        if os.path.exists(path):
            return path
    return None

font_path = find_korean_font()
_fp = fm.FontProperties(fname=font_path) if font_path else fm.FontProperties()
plt.rcParams['axes.unicode_minus'] = False

# ── 그래프 정의 ──────────────────────────────────────────────
pos = {
    '트리토리아진':    (-2.5,  1.5),
    '다온카츠':        (-1.0,  0.0),
    '익선취향':        ( 0.5,  0.0),
    '마복림할머니집':  ( 0.5, -1.8),
    '이쿠':            ( 2.0, -1.8),
    '땀땀':            ( 3.2, -0.5),
    '담금':            ( 3.2,  1.5),
    '빠오즈푸':        ( 1.2,  1.5),
}

# 수정된 간선 리스트
edges = [
    ('마복림할머니집', '빠오즈푸',      70),
    ('트리토리아진', '빠오즈푸',       208),
    ('트리토리아진', '다온카츠',       249),
    ('다온카츠',     '익선취향',       164),
    ('익선취향',     '마복림할머니집',  52),
    ('익선취향',     '빠오즈푸',       119),
    ('마복림할머니집','이쿠',          222),
    ('빠오즈푸',     '담금',           141),
    ('이쿠',         '땀땀',           120),
    ('땀땀',         '담금',           173),
    ('빠오즈푸',        '땀땀',         115)
]

graph = {node: {} for node in pos}
for u, v, w in edges:
    graph[u][v] = w
    graph[v][u] = w

nodes = list(pos.keys())

# 간선 가중치 쉽게 조회하기 위한 딕셔너리
edge_weights = {}
for u, v, w in edges:
    edge_weights[(u, v)] = w
    edge_weights[(v, u)] = w

# ── 프림 알고리즘 스텝 기록 ───────────────────────────────────
def prim_steps(graph, start):
    visited = [start]
    visited_set = {start}
    mst_edges = []
    steps = []

    # 초기 상태 후보 계산
    candidates = []
    for v, w in graph[start].items():
        candidates.append((w, start, v))
    candidates.sort(key=lambda x: x[0])

    steps.append({
        'visited':      list(visited),
        'mst_edges':    list(mst_edges),
        'current_node': start,
        'candidates':   [f"{u} - {v} ({w}분)" for w, u, v in candidates]
    })

    while len(visited_set) < len(graph):
        # 방문한 노드 집합과 방문하지 않은 노드 집합을 연결하는 모든 간선 수집
        candidates = []
        for u in visited_set:
            for v, w in graph[u].items():
                if v not in visited_set:
                    candidates.append((w, u, v))

        if not candidates:
            break

        # 가중치가 가장 작은 간선 선택 (우선순위 큐 역할)
        candidates.sort(key=lambda x: x[0])
        w, u, v = candidates[0]

        mst_edges.append((u, v))
        visited.append(v)
        visited_set.add(v)

        # 다음 루프를 위한 후보군 목록 정리
        next_candidates = []
        for u_n in visited_set:
            for v_n, w_n in graph[u_n].items():
                if v_n not in visited_set:
                    next_candidates.append((w_n, u_n, v_n))
        next_candidates.sort(key=lambda x: x[0])

        steps.append({
            'visited':      list(visited),
            'mst_edges':    list(mst_edges),
            'current_node': v,
            'candidates':   [f"{u_c} - {v_c} ({w_c}분)" for w_c, u_c, v_c in next_candidates]
        })

    return steps

steps = prim_steps(graph, '트리토리아진')
all_steps = steps + [steps[-1]] * 30

# ── 색상 ─────────────────────────────────────────────────────
C_BG        = '#1a1a2e'
C_EDGE_DEF  = '#4a4a6a'
C_EDGE_MST  = '#e94560'  # MST 선택 간선 (두꺼운 실선)
C_EDGE_CAND = '#533483'  # 후보 간선 (점선)
C_NODE_DEF  = '#16213e'
C_NODE_VIS  = '#0f3460'
C_NODE_CUR  = '#e94560'
C_TEXT      = '#eaeaea'
C_WT        = '#a0a0c0'

# ── Figure 설정 ──────────────────────────────────────────────
fig = plt.figure(figsize=(14, 8), facecolor=C_BG)
ax_graph = fig.add_axes([0.0, 0.12, 0.72, 0.82])
ax_info  = fig.add_axes([0.73, 0.12, 0.25, 0.82])

NODE_RADIUS = 0.18

def draw_frame(i):
    ax_graph.cla()
    ax_info.cla()
    for ax in [ax_graph, ax_info]:
        ax.set_facecolor(C_BG)
        ax.axis('off')

    step         = all_steps[i]
    visited      = step['visited']
    mst_edges    = step['mst_edges']
    current_node = step['current_node']
    candidates   = step['candidates']

    # 현재 총 MST 가중치 계산
    total_weight = sum(edge_weights[e] for e in mst_edges)

    # x, y 범위 자동 설정
    xs = [p[0] for p in pos.values()]
    ys = [p[1] for p in pos.values()]
    ax_graph.set_xlim(min(xs) - 0.8, max(xs) + 0.8)
    ax_graph.set_ylim(min(ys) - 0.8, max(ys) + 0.8)
    ax_graph.set_aspect('equal')

    # 간선 그리기
    for u, v, w in edges:
        x1, y1 = pos[u]
        x2, y2 = pos[v]

        is_mst = (u, v) in mst_edges or (v, u) in mst_edges
        is_candidate = not is_mst and ((u in visited and v not in visited) or (v in visited and u not in visited))

        if is_mst:
            color = C_EDGE_MST
            lw = 3.2
            ls = '-'
        elif is_candidate:
            color = C_EDGE_CAND
            lw = 1.8
            ls = '--'
        else:
            color = C_EDGE_DEF
            lw = 1.0
            ls = '-'

        ax_graph.plot([x1, x2], [y1, y2], color=color, linewidth=lw, linestyle=ls, zorder=1)

        # 가중치 라벨
        mx, my = (x1 + x2) / 2, (y1 + y2) / 2
        ax_graph.text(mx, my, f'{w}분', color=C_WT, fontsize=7.5,
                      ha='center', va='center', fontproperties=_fp, zorder=3,
                      bbox=dict(boxstyle='round,pad=0.2', fc=C_BG, ec='none', alpha=0.8))

    # 노드 그리기
    for node in nodes:
        x, y = pos[node]
        if node == current_node:
            color = C_NODE_CUR
        elif node in visited:
            color = C_NODE_VIS
        else:
            color = C_NODE_DEF

        circle = plt.Circle((x, y), NODE_RADIUS, color=color,
                             ec='white', linewidth=1.5, zorder=2)
        ax_graph.add_patch(circle)
        ax_graph.text(x, y, node, color=C_TEXT, fontsize=8,
                      ha='center', va='center', fontproperties=_fp, zorder=4,
                      fontweight='bold' if node == current_node else 'normal')

    ax_graph.set_title('Prim 알고리즘 — 최소 신장 트리(MST) 생성', color=C_TEXT,
                       fontsize=14, fontweight='bold', pad=8, fontproperties=_fp)

    # ── 정보 패널 ──────────────────────────────────────────
    ax_info.set_xlim(0, 1)
    ax_info.set_ylim(0, 1)

    legend_items = [
        (C_NODE_CUR, '최근 추가된 노드'),
        (C_NODE_VIS, 'MST 포함 노드'),
        (C_NODE_DEF, '미방문 노드'),
    ]
    ax_info.text(0.05, 0.97, '범례', color=C_TEXT, fontsize=10,
                 fontweight='bold', va='top', fontproperties=_fp)
    for k, (color, label) in enumerate(legend_items):
        y = 0.91 - k * 0.055
        ax_info.add_patch(plt.Circle((0.08, y), 0.024, color=color))
        ax_info.text(0.18, y, label, color=C_TEXT, fontsize=8.5,
                     va='center', fontproperties=_fp)

    # 총 소요 시간 (MST 비용) 표시
    ax_info.text(0.05, 0.72, f'총 대기 시간: {total_weight}분',
                 color='#00ffcc', fontsize=11, fontweight='bold', va='top', fontproperties=_fp)

    # MST 선택 간선 출력
    ax_info.text(0.05, 0.64, f'선택된 MST 간선 ({len(mst_edges)}/{len(nodes)-1})',
                 color=C_TEXT, fontsize=9.5, fontweight='bold', va='top', fontproperties=_fp)
    for k, (u, v) in enumerate(mst_edges):
        y = 0.59 - k * 0.045
        w = edge_weights[(u, v)]
        ax_info.text(0.05, y, f'{k+1}. {u} - {v} ({w}분)', color=C_EDGE_MST,
                     fontsize=8, va='top', fontproperties=_fp)

    # 우선순위 큐 대기열 출력
    ax_info.text(0.05, 0.22, '후보 간선 대기열 (Top 3)', color=C_TEXT, fontsize=9,
                 fontweight='bold', va='top', fontproperties=_fp)
    cand_str = '\n'.join(candidates[:3]) if candidates else '(MST 완성)'
    ax_info.text(0.05, 0.17, cand_str, color='#aaaaff', fontsize=8,
                 va='top', fontproperties=_fp)

    fig.text(0.01, 0.01, f'Step {min(i+1, len(steps))}/{len(steps)}',
             color='#666688', fontsize=8, ha='left')


ani = animation.FuncAnimation(
    fig, draw_frame,
    frames=len(all_steps),
    interval=1200,  # 과정을 더 잘 관찰할 수 있도록 인터벌을 살짝 조정
    repeat=False
)

plt.show()

In [ ]:
import os
import matplotlib
matplotlib.use('TkAgg')  # 환경에 맞게 변경
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.animation as animation
import matplotlib.font_manager as fm

# ── 한글 폰트 설정 ──────────────────────────────────────────
def find_korean_font():
    candidates = [
        'C:/Windows/Fonts/malgun.ttf',
        'C:/Windows/Fonts/gulim.ttc',
        '/System/Library/Fonts/AppleSDGothicNeo.ttc',
        '/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
        '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc',
    ]
    for path in candidates:
        if os.path.exists(path):
            return path
    return None

font_path = find_korean_font()
_fp = fm.FontProperties(fname=font_path) if font_path else fm.FontProperties()
plt.rcParams['axes.unicode_minus'] = False

# ── 그래프 정의 ──────────────────────────────────────────────
pos = {
    '트리토리아진':    (-2.5,  1.5),
    '다온카츠':        (-1.0,  0.0),
    '익선취향':        ( 0.5,  0.0),
    '마복림할머니집':  ( 0.5, -1.8),
    '이쿠':            ( 2.0, -1.8),
    '땀땀':            ( 3.2, -0.5),
    '담금':            ( 3.2,  1.5),
    '빠오즈푸':        ( 1.2,  1.5),
}

# 수정된 간선 리스트
edges = [
    ('마복림할머니집', '빠오즈푸',      70),
    ('트리토리아진', '빠오즈푸',       208),
    ('트리토리아진', '다온카츠',       249),
    ('다온카츠',     '익선취향',       164),
    ('익선취향',     '마복림할머니집',  52),
    ('익선취향',     '빠오즈푸',       119),
    ('마복림할머니집','이쿠',          222),
    ('빠오즈푸',     '담금',           141),
    ('이쿠',         '땀땀',           120),
    ('땀땀',         '담금',           173),
    ('빠오즈푸',        '땀땀',         115)
]

nodes = list(pos.keys())

# ── Union-Find (서로소 집합) 구현 ──────────────────────────────
parent = {node: node for node in nodes}

def find(i):
    if parent[i] == i:
        return i
    parent[i] = find(parent[i])
    return parent[i]

def union(i, j):
    root_i = find(i)
    root_j = find(j)
    if root_i != root_j:
        parent[root_i] = root_j
        return True
    return False

# ── 크루스칼 알고리즘 스텝 기록 ─────────────────────────────────
def kruskal_steps(edges, nodes):
    # 간선 가중치 기준 오름차순 정렬
    sorted_edges = sorted(edges, key=lambda x: x[2])

    mst_edges = []
    rejected_edges = []
    steps = []

    # 각 간선을 하나씩 검사하는 과정을 기록
    for u, v, w in sorted_edges:
        current_edge = (u, v, w)

        # 사이클 검사 (부모가 같으면 사이클 형성)
        if find(u) != find(v):
            union(u, v)
            status = 'Accepted'
            mst_edges.append((u, v, w))
        else:
            status = 'Rejected'
            rejected_edges.append((u, v, w))

        # 우측 패널에 띄울 간선들의 상태 리스트 생성
        edge_status_list = []
        for e_u, e_v, e_w in sorted_edges:
            if (e_u, e_v, e_w) == current_edge:
                state = f"▶ {e_u}-{e_v} ({e_w}분)"
            elif any(e_u==m[0] and e_v==m[1] for m in mst_edges):
                state = f"✅ {e_u}-{e_v} ({e_w}분)"
            elif any(e_u==r[0] and e_v==r[1] for r in rejected_edges):
                state = f"❌ {e_u}-{e_v} ({e_w}분)"
            else:
                state = f"   {e_u}-{e_v} ({e_w}분)"
            edge_status_list.append((state, (e_u, e_v, e_w) == current_edge, status if (e_u, e_v, e_w) == current_edge else None))

        # 현재까지 생성된 그룹(컴포넌트)에 포함된 노드들 파악
        active_nodes = set()
        for m_u, m_v, _ in mst_edges:
            active_nodes.add(m_u)
            active_nodes.add(m_v)

        steps.append({
            'current_edge': current_edge,
            'status': status,
            'mst_edges': list(mst_edges),
            'rejected_edges': list(rejected_edges),
            'edge_status_list': edge_status_list,
            'active_nodes': active_nodes
        })

    return steps

steps = kruskal_steps(edges, nodes)
all_steps = steps + [steps[-1]] * 25

# ── 색상 ─────────────────────────────────────────────────────
C_BG        = '#1a1a2e'
C_EDGE_DEF  = '#4a4a6a'
C_EDGE_MST  = '#e94560'  # MST로 확정된 간선 (두꺼운 실선)
C_EDGE_CUR  = '#00fff0'  # 현재 검사 중인 간선 (하늘색)
C_EDGE_REJ  = '#333344'  # 탈락한 간선 (흐린 회색)
C_NODE_DEF  = '#16213e'
C_NODE_ACT  = '#0f3460'  # MST 서브트리에 연결된 노드
C_NODE_CUR  = '#e94560'  # 현재 검as중인 간선에 포함된 노드
C_TEXT      = '#eaeaea'
C_WT        = '#a0a0c0'

# ── Figure 설정 ──────────────────────────────────────────────
fig = plt.figure(figsize=(14, 8), facecolor=C_BG)
ax_graph = fig.add_axes([0.0, 0.12, 0.70, 0.82])
ax_info  = fig.add_axes([0.71, 0.12, 0.28, 0.82])

NODE_RADIUS = 0.18

def draw_frame(i):
    ax_graph.cla()
    ax_info.cla()
    for ax in [ax_graph, ax_info]:
        ax.set_facecolor(C_BG)
        ax.axis('off')

    step             = all_steps[i]
    current_edge     = step['current_edge']
    status           = step['status']
    mst_edges        = step['mst_edges']
    rejected_edges   = step['rejected_edges']
    edge_status_list = step['edge_status_list']
    active_nodes     = step['active_nodes']

    total_weight = sum(e[2] for e in mst_edges)

    # x, y 범위 자동 설정
    xs = [p[0] for p in pos.values()]
    ys = [p[1] for p in pos.values()]
    ax_graph.set_xlim(min(xs) - 0.8, max(xs) + 0.8)
    ax_graph.set_ylim(min(ys) - 0.8, max(ys) + 0.8)
    ax_graph.set_aspect('equal')

    # 간선 그리기
    for u, v, w in edges:
        x1, y1 = pos[u]
        x2, y2 = pos[v]

        is_current = (u == current_edge[0] and v == current_edge[1]) or (v == current_edge[0] and u == current_edge[1])
        is_mst = any((u==e[0] and v==e[1]) or (v==e[0] and u==e[1]) for e in mst_edges)
        is_rejected = any((u==e[0] and v==e[1]) or (v==e[0] and u==e[1]) for e in rejected_edges)

        if is_current:
            color = C_EDGE_CUR
            lw = 4.0
            ls = '-'
        elif is_mst:
            color = C_EDGE_MST
            lw = 3.2
            ls = '-'
        elif is_rejected:
            color = C_EDGE_REJ
            lw = 1.0
            ls = ':'
        else:
            color = C_EDGE_DEF
            lw = 1.2
            ls = '-'

        ax_graph.plot([x1, x2], [y1, y2], color=color, linewidth=lw, linestyle=ls, zorder=1)

        # 가중치 라벨
        mx, my = (x1 + x2) / 2, (y1 + y2) / 2
        ax_graph.text(mx, my, f'{w}분', color=C_WT, fontsize=7.5,
                      ha='center', va='center', fontproperties=_fp, zorder=3,
                      bbox=dict(boxstyle='round,pad=0.2', fc=C_BG, ec='none', alpha=0.8))

    # 노드 그리기
    for node in nodes:
        x, y = pos[node]
        if node in (current_edge[0], current_edge[1]):
            color = C_NODE_CUR
        elif node in active_nodes:
            color = C_NODE_ACT
        else:
            color = C_NODE_DEF

        circle = plt.Circle((x, y), NODE_RADIUS, color=color,
                             ec='white', linewidth=1.5, zorder=2)
        ax_graph.add_patch(circle)
        ax_graph.text(x, y, node, color=C_TEXT, fontsize=8,
                      ha='center', va='center', fontproperties=_fp, zorder=4,
                      fontweight='bold' if node in (current_edge[0], current_edge[1]) else 'normal')

    ax_graph.set_title('Kruskal 알고리즘 — 최소 신장 트리(MST) 생성', color=C_TEXT,
                       fontsize=14, fontweight='bold', pad=8, fontproperties=_fp)

    # ── 정보 패널 ──────────────────────────────────────────
    ax_info.set_xlim(0, 1)
    ax_info.set_ylim(0, 1)

    legend_items = [
        (C_EDGE_CUR, '현재 검사 중인 간선'),
        (C_EDGE_MST, '선택된 MST 간선'),
        (C_EDGE_REJ, '탈락된 간선 (사이클)'),
    ]
    ax_info.text(0.02, 0.97, '범례', color=C_TEXT, fontsize=10,
                 fontweight='bold', va='top', fontproperties=_fp)
    for k, (color, label) in enumerate(legend_items):
        y = 0.91 - k * 0.05
        ax_info.plot([0.04, 0.12], [y, y], color=color, linewidth=3 if color != C_EDGE_REJ else 1,
                     linestyle=':' if color == C_EDGE_REJ else '-')
        ax_info.text(0.16, y, label, color=C_TEXT, fontsize=8.5, va='center', fontproperties=_fp)

    # 총 소요 시간 (MST 비용) 표시
    ax_info.text(0.02, 0.74, f'총 대기 시간: {total_weight}분',
                 color='#00ffcc', fontsize=11, fontweight='bold', va='top', fontproperties=_fp)

    # 크루스칼 간선 정렬 현황 및 진행 상황 출력
    ax_info.text(0.02, 0.67, '간선 정렬 및 처리 상태 (가중치 순)',
                 color=C_TEXT, fontsize=9.5, fontweight='bold', va='top', fontproperties=_fp)

    for k, (text, is_curr, edge_status) in enumerate(edge_status_list):
        y = 0.61 - k * 0.05
        if is_curr:
            text_color = C_EDGE_CUR
            if edge_status == 'Accepted':
                text += " -> 선택!"
            else:
                text += " -> 사이클 탈락!"
        elif '✅' in text:
            text_color = '#ff79c6'
        elif '❌' in text:
            text_color = '#6272a4'
        else:
            text_color = '#8be9fd'

        ax_info.text(0.02, y, text, color=text_color, fontsize=8, va='top', fontproperties=_fp)

    fig.text(0.01, 0.01, f'Step {min(i+1, len(steps))}/{len(steps)}',
             color='#666688', fontsize=8, ha='left')


ani = animation.FuncAnimation(
    fig, draw_frame,
    frames=len(all_steps),
    interval=1500,  # 각 간선의 채택 여부를 확실히 인지할 수 있도록 인터벌 부여
    repeat=False
)

plt.show()